In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, make_scorer, fbeta_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Define features and target variable
features = [
    # encoded station/service
    'StationCode_TE',
    'Service:Type_Intercity',
    'Service:Type_Intercity direct',
    'Service:Type_Sprinter',
    # temporal
    'Hour_sin',
    'Hour_cos',
    'DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'IsWeekend',
    'RushHour',
    # operational context
    'StationTraffic',
    'Stop:Platform change',
    'arrival_scheduled',
    'departure_scheduled',
    # weather
    'Wind Direction',
    'Hourly Mean Wind Speed',
    'Wind Speed last 10 Minutes',
    'Max Wind Speed',
    'Temperature',
    'Dew Point temperature',
    'Sunshine Duration',
    'Global Radiation',
    'Precipitation Duration',
    'Precipitation Amount',
    'Air Pressure',
    'Horizontal Visibility',
    'Cloud Cover',
    'Humidity',
    'Fog',
    'Rainfall',
    'Snowfall',
    'Thunder',
    'Hail',
]

target = "is_cancelled"

In [ ]:
# Prepare for chunked reading
cols = features + [target]
chunk_size = 1_000_000
sample_size = 1_000_000
random_state = 42

dtype_map = {col: "float32" for col in features}
dtype_map[target] = "int8"


# Read CSV in chunks
def chunk_reader(file_path):
    for chunk in pd.read_csv(
        file_path,
        usecols=cols,
        dtype=dtype_map,
        chunksize=chunk_size
    ):
        chunk = chunk.reindex(columns=cols, fill_value=0) # Ensure all columns are present

        X_chunk = chunk[features]
        y_chunk = chunk[target]

        yield X_chunk, y_chunk # Yield the chunk for processing
        
# Count classes
not_cancelled_total = 0
cancelled_total = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):
    not_cancelled_total += (y_chunk == 0).sum()
    cancelled_total += (y_chunk == 1).sum()

total_rows = not_cancelled_total + cancelled_total

print(f"Train not cancelled: {not_cancelled_total:,}")
print(f"Train cancelled: {cancelled_total:,}")
print(f"Train total:     {total_rows:,}")

# Decide sample sizes
not_cancelled_sample_size = int(sample_size * not_cancelled_total / total_rows)
cancelled_sample_size = sample_size - not_cancelled_sample_size

print(f"Sampling not cancelled: {not_cancelled_sample_size:,}")
print(f"Sampling cancelled: {cancelled_sample_size:,}")


Train chunks:
Train chunk 1: (1000000, 26)
Train chunk 2: (1000000, 26)
Train chunk 3: (1000000, 26)
Train chunk 4: (1000000, 26)
Train chunk 5: (1000000, 26)
Train chunk 6: (1000000, 26)
Train chunk 7: (1000000, 26)
Train chunk 8: (1000000, 26)
Train chunk 9: (1000000, 26)
Train chunk 10: (1000000, 26)
Train chunk 11: (1000000, 26)
Train chunk 12: (1000000, 26)
Train chunk 13: (1000000, 26)
Train chunk 14: (1000000, 26)
Train chunk 15: (1000000, 26)
Train chunk 16: (1000000, 26)
Train chunk 17: (1000000, 26)
Train chunk 18: (1000000, 26)
Train chunk 19: (1000000, 26)
Train chunk 20: (1000000, 26)
Train chunk 21: (1000000, 26)
Train chunk 22: (1000000, 26)
Train chunk 23: (1000000, 26)
Train chunk 24: (1000000, 26)
Train chunk 25: (1000000, 26)
Train chunk 26: (1000000, 26)
Train chunk 27: (1000000, 26)
Train chunk 28: (1000000, 26)
Train chunk 29: (1000000, 26)
Train chunk 30: (1000000, 26)
Train chunk 31: (1000000, 26)
Train chunk 32: (1000000, 26)
Train chunk 33: (1000000, 26)
Trai

In [ ]:
# Generate random numbers for sampling (default_rng is faster)
rng = np.random.default_rng(random_state)

X_not_cancelled_parts = []
y_not_cancelled_parts = []
X_cancelled_parts = []
y_cancelled_parts = []

not_cancelled_seen = 0
cancelled_seen = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):

    # Create masks for cancelled and not cancelled
    not_cancelled_mask = y_chunk == 0
    cancelled_mask = y_chunk == 1
    
    # Split the chunk into cancelled and not cancelled
    X_not_cancelled = X_chunk.loc[not_cancelled_mask]
    y_not_cancelled = y_chunk.loc[not_cancelled_mask]

    X_cancelled = X_chunk.loc[cancelled_mask]
    y_cancelled = y_chunk.loc[cancelled_mask]

    # sample fraction based on remaining needed / remaining available
    neg_remaining_needed = not_cancelled_sample_size - sum(len(p) for p in y_not_cancelled_parts)
    pos_remaining_needed = cancelled_sample_size - sum(len(p) for p in y_cancelled_parts)

    neg_remaining_total = not_cancelled_total - not_cancelled_seen
    pos_remaining_total = cancelled_total - cancelled_seen

    # Sample not cancelled
    if neg_remaining_needed > 0 and len(X_not_cancelled) > 0:
        p_neg = min(1.0, neg_remaining_needed / max(neg_remaining_total, 1))
        keep_neg = rng.random(len(X_not_cancelled)) < p_neg

        X_not_cancelled_parts.append(X_not_cancelled.loc[keep_neg])
        y_not_cancelled_parts.append(y_not_cancelled.loc[keep_neg])
        
    # Sample cancelled
    if pos_remaining_needed > 0 and len(X_cancelled) > 0:
        p_pos = min(1.0, pos_remaining_needed / max(pos_remaining_total, 1))
        keep_pos = rng.random(len(X_cancelled)) < p_pos

        X_cancelled_parts.append(X_cancelled.loc[keep_pos])
        y_cancelled_parts.append(y_cancelled.loc[keep_pos])

    not_cancelled_seen += len(X_not_cancelled)
    cancelled_seen += len(X_cancelled)

# Combine sampled parts
X_train_sample = pd.concat(X_not_cancelled_parts + X_cancelled_parts, axis=0)
y_train_sample = pd.concat(y_not_cancelled_parts + y_cancelled_parts, axis=0)

# Shuffle final sample
shuffle_idx = rng.permutation(len(X_train_sample))
X_train_sample = X_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.float32, copy=False)
y_train_sample = y_train_sample.iloc[shuffle_idx].to_numpy(copy=False)

In [ ]:
# Handle class imbalance
neg_count = (y_train_sample == 0).sum()
pos_count = (y_train_sample == 1).sum()

scale_pos_weight = neg_count / max(pos_count, 1)

print(f"scale_pos_weight: {scale_pos_weight:.4f}")

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_sample, y_train_sample)
print(f"Random Forest trained on {len(y_train_sample):,} samples (stratified)")

Random Forest trained on 500,000 samples (stratified)
F2 Score: 0.238

Validation
              precision    recall  f1-score   support

           0      0.955     0.737     0.832   9439441
           1      0.087     0.422     0.144    560559

    accuracy                          0.719  10000000
   macro avg      0.521     0.579     0.488  10000000
weighted avg      0.907     0.719     0.793  10000000

ROC-AUC: 0.6071
F2 Score: 0.1839

Test
              precision    recall  f1-score   support

           0      0.931     0.867     0.898   8426700
           1      0.117     0.214     0.151    691095

    accuracy                          0.818   9117795
   macro avg      0.524     0.541     0.525   9117795
weighted avg      0.869     0.818     0.841   9117795

ROC-AUC: 0.601

Top 20 Feature Importances:
Service:Type_Sprinter     0.133774
Service:Type_Intercity    0.080499
Air Pressure              0.077206
Temperature               0.064967
Max Wind Speed            0.057786
Month_

In [ ]:
# Evaluate CSV split in chunks
def evaluate_split_xgb(name, file_path):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(xgb.predict(X_chunk_np))
        y_prob_parts.append(xgb.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)

    print(f"\n{name}")
    print("F2 Score:", round(f2, 4))
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(average_precision_score(y_true_all, y_prob_all), 4))


evaluate_split_xgb("Validation", "val_data.csv")
evaluate_split_xgb("Test", "test_data.csv")